In [10]:
# Cell 1: Setup Paths, Shared Normalization Import, and Constants
import sys
from pathlib import Path

# تنظیم مسیر ریشه پروژه جهت دسترسی به پکیج src
current_dir = Path.cwd()
project_root = current_dir if (current_dir / "src").exists() else current_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if (project_root / "src").exists() and str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

# ایمپورت تابع استاندارد نرمال‌سازی فارسی که در نوت‌بوک 03 استفاده شده بود
try:
    from src.digikala_ai.utils.normalization import normalize_persian_text
    print("✓ Successfully imported 'normalize_persian_text' from src.digikala_ai.utils.normalization")
except ImportError:
    try:
        from normalization import normalize_persian_text
        print("✓ Successfully imported 'normalize_persian_text' from local normalization.py")
    except ImportError:
        raise ImportError("Could not find 'normalization.py'. Please verify your sys.path or project structure.")

import csv
import random
from collections import Counter
from time import perf_counter

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline

# متغیرهای ثابت
LABELS = ['not_recommended', 'no_idea', 'recommended']
CAP = 500000
SEED = 42

random.seed(SEED)

# مسیر کاندید فایل نظرات
candidate_paths = [
    Path("../digikala/digikala-comments.csv"),
    Path("digikala/digikala-comments.csv"),
    Path("../../digikala/digikala-comments.csv"),
    project_root / "digikala" / "digikala-comments.csv"
]

comments_path = None
for p in candidate_paths:
    if p.exists():
        comments_path = p.resolve()
        break

if comments_path is None:
    comments_path = Path("digikala/digikala-comments.csv")

print(f"Comments path resolved to: {comments_path}")
print(f"Target Labels: {LABELS}")
print(f"Sampling Cap: {CAP:,} | Random Seed: {SEED}")


✓ Successfully imported 'normalize_persian_text' from src.digikala_ai.utils.normalization
Comments path resolved to: D:\goal\programming\quera\boot_camp\project3\digikala\digikala-comments.csv
Target Labels: ['not_recommended', 'no_idea', 'recommended']
Sampling Cap: 500,000 | Random Seed: 42


In [11]:
# Cell 2: Comment-level Preprocessing & Reservoir Sampling
def clean_comment_field(value: str) -> str:
    """پوشش نازک روی نرمال‌سازی مشترک پروژه جهت کنترل مقادیر تهی و NaN"""
    if not value or str(value).strip().lower() == "nan":
        return ""
    # فراخوانی تابع اصلی از نوت‌بوک 03 / normalization.py
    norm = normalize_persian_text(str(value))
    return "" if norm.lower() == "nan" else norm

print(f"Reading and sampling dataset from: {comments_path} ...")
t0 = perf_counter()

pool = []
seen = 0

with open(comments_path, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        label = (row.get("recommendation_status") or "").strip()
        if label not in LABELS:
            continue
        seen += 1
        # الگوریتم نمونه‌گیری با مخزن (Reservoir Sampling)
        if len(pool) < CAP:
            pool.append(row)
        else:
            j = random.randint(0, seen - 1)
            if j < CAP:
                pool[j] = row

print(f"Total eligible rows processed: {seen:,}")
print(f"Sampled rows in pool: {len(pool):,} (took {perf_counter() - t0:.2f}s)")


Reading and sampling dataset from: D:\goal\programming\quera\boot_camp\project3\digikala\digikala-comments.csv ...
Total eligible rows processed: 5,261,863
Sampled rows in pool: 500,000 (took 30.32s)


In [12]:
# Cell 3: Text Merging and Strict Deduplication
cleaned_records = []
for row in pool:
    cid = (row.get("id") or "").strip()
    pid = (row.get("product_id") or "").strip()
    title = clean_comment_field(row.get("title") or "")
    body = clean_comment_field(row.get("body") or "")
    text = f"{title} {body}".strip()
    label = row["recommendation_status"].strip()
    
    if cid and pid and text and label in LABELS:
        cleaned_records.append((cid, pid, text, label))

# شناسایی و حذف موارد تکراری بر اساس شناسه نظر و متن نرمال‌شده
id_counts = Counter(r[0] for r in cleaned_records)
text_counts = Counter(r[2] for r in cleaned_records)

filtered_records = [
    r for r in cleaned_records
    if id_counts[r[0]] == 1 and text_counts[r[2]] == 1
]

print(f"Records before deduplication: {len(cleaned_records):,}")
print(f"Unique clean records retained: {len(filtered_records):,}")

texts = [r[2] for r in filtered_records]
labels = [r[3] for r in filtered_records]
groups = [r[1] for r in filtered_records]


Records before deduplication: 499,997
Unique clean records retained: 396,727


In [13]:
# Cell 4: Group-aware Split (Train 70% | Validation 15% | Test 15%)
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_val_idx, test_idx = next(gss_outer.split(texts, labels, groups=groups))

val_rel_size = 0.15 / 0.85
gss_inner = GroupShuffleSplit(n_splits=1, test_size=val_rel_size, random_state=SEED)

train_val_texts = [texts[i] for i in train_val_idx]
train_val_labels = [labels[i] for i in train_val_idx]
train_val_groups = [groups[i] for i in train_val_idx]

train_sub_idx, val_sub_idx = next(gss_inner.split(train_val_texts, train_val_labels, groups=train_val_groups))

train_texts = [train_val_texts[i] for i in train_sub_idx]
train_labels = [train_val_labels[i] for i in train_sub_idx]

val_texts = [train_val_texts[i] for i in val_sub_idx]
val_labels = [train_val_labels[i] for i in val_sub_idx]

# بررسی اینکه هر سه کلاس در داده‌ها وجود داشته باشند
assert set(train_labels) == set(LABELS), "Missing labels in Train split"
assert set(val_labels) == set(LABELS), "Missing labels in Validation split"

print(f"Train set: {len(train_texts):,} samples | Distribution: {Counter(train_labels)}")
print(f"Validation set: {len(val_texts):,} samples | Distribution: {Counter(val_labels)}")
print(f"Test set (reserved): {len(test_idx):,} samples (Held out)")


Train set: 278,295 samples | Distribution: Counter({'recommended': 212853, 'no_idea': 33731, 'not_recommended': 31711})
Validation set: 58,888 samples | Distribution: Counter({'recommended': 45072, 'no_idea': 7076, 'not_recommended': 6740})
Test set (reserved): 59,544 samples (Held out)


In [14]:
# Cell 5: Train and Evaluate Baseline Models on CPU
# models = {
#     "Majority Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
#     "TF-IDF + Logistic Regression": make_pipeline(
#         TfidfVectorizer(min_df=2, max_features=40000),
#         LogisticRegression(max_iter=400, class_weight="balanced", random_state=SEED)
#     )
# }
models = {
    "Majority Baseline": DummyClassifier(strategy="most_frequent"),
    "TF-IDF + Logistic Regression (Optimized N-gram)": make_pipeline(
        TfidfVectorizer(
            min_df=2, 
            max_features=60000, 
            ngram_range=(1, 2)  # اضافه کردن دوکلمه‌ها برای ثبت عباراتی مثل «اصلا خوب نیست»
        ),
        LogisticRegression(
            max_iter=500, 
            C=1.2,              # تنظیم ضریب جریمه برای دقت بیشتر روی کلاس‌های سخت مثل no_idea
            class_weight="balanced", 
            random_state=SEED
        )
    )
}


trained_models = {}

for name, model in models.items():
    print("=" * 60)
    print(f"Training: {name}")
    t_start = perf_counter()
    
    model.fit(train_texts, train_labels)
    fit_time = perf_counter() - t_start
    
    t_pred = perf_counter()
    val_preds = model.predict(val_texts)
    pred_time = perf_counter() - t_pred
    
    macro_f1 = f1_score(val_labels, val_preds, labels=LABELS, average="macro")
    trained_models[name] = model
    
    print(f"Fit Time: {fit_time:.2f}s | Predict Time: {pred_time:.4f}s")
    print(f">> Validation Macro-F1: {macro_f1:.4f}")
    
    if name == "TF-IDF + Logistic Regression":
        print("\n--- Detailed Classification Report ---")
        print(classification_report(val_labels, val_preds, labels=LABELS, digits=4))
        print("--- Confusion Matrix ---")
        print(f"Labels order: {LABELS}")
        print(confusion_matrix(val_labels, val_preds, labels=LABELS))


Training: Majority Baseline
Fit Time: 0.21s | Predict Time: 0.0015s
>> Validation Macro-F1: 0.2890
Training: TF-IDF + Logistic Regression (Optimized N-gram)
Fit Time: 25.50s | Predict Time: 1.0757s
>> Validation Macro-F1: 0.6807


In [15]:
# Cell 6: Export Trained Pipeline for ml_service.py
import joblib

best_pipeline = trained_models.get("TF-IDF + Logistic Regression (Optimized N-gram)")

export_dir = project_root / "m2_outputs"
export_dir.mkdir(parents=True, exist_ok=True)
model_export_path = export_dir / "best_model.joblib"

if best_pipeline is not None:
    joblib.dump(best_pipeline, model_export_path)
    print(f"✓ Pipeline exported to: {model_export_path.resolve()}")
    print("This pipeline contains both the TF-IDF vectorizer and the classifier.")


✓ Pipeline exported to: D:\goal\programming\quera\boot_camp\project3\notebook\m2_outputs\best_model.joblib
This pipeline contains both the TF-IDF vectorizer and the classifier.


In [16]:
# Cell 7: Hardware & CUDA Readiness Check (RTX 3050 - 4GB)
import torch

print("=" * 60)
print("GPU & PyTorch Verification for Future Transformer Benchmarks")
print("=" * 60)

cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    device_id = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device_id)
    total_vram_gb = torch.cuda.get_device_properties(device_id).total_memory / (1024 ** 3)
    
    print(f"Active GPU: {gpu_name}")
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
    
    # تست سبک اجرای تانسور روی GPU
    x = torch.randn((1000, 1000), device="cuda")
    y = torch.matmul(x, x)
    torch.cuda.synchronize()
    print("✓ Quick Tensor Benchmark on RTX 3050: PASSED successfully.")
    print("Status: Hardware is ready for Transformer fine-tuning / embedding extraction.")
else:
    print("⚠ CUDA not detected. Ensure NVIDIA drivers and PyTorch CUDA build are properly configured.")


GPU & PyTorch Verification for Future Transformer Benchmarks
CUDA Available: True
Active GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Total VRAM: 4.00 GB
✓ Quick Tensor Benchmark on RTX 3050: PASSED successfully.
Status: Hardware is ready for Transformer fine-tuning / embedding extraction.
